# PyDBAdminKit 0.3.0 — Notebook de démonstration

> Ce notebook couvre l'ensemble des fonctionnalités de `pydbadminkit` via l'API Python et la CLI.  
> Il utilise **PostgreSQL local** (profil `local-native`) mais est compatible avec Docker (profil `local`).

**Prérequis :**
- `pip install -e ".[dev,binary]"` depuis la racine du projet
- PostgreSQL démarré (local ou Docker)
- Variable d'environnement définie (voir Cellule 2)

---

## Table des matières
1. [Configuration de l'environnement](#1)
2. [Foundation — Connexion et capabilities](#2)
3. [Object Explorer — Server & Databases](#3)
4. [Object Explorer — Schemas, Tables, Vues, Index](#4)
5. [Security — Inspection des rôles](#5)
6. [Security — Accès directs et effectifs](#6)
7. [Security — Ownership](#7)
8. [Mutations — Dry-run & création de rôles](#8)
9. [Mutations — Memberships & GRANT/REVOKE](#9)
10. [Sortie machine — JSON & YAML](#10)
11. [Nettoyage](#11)

---
## 1. Configuration de l'environnement <a id='1'></a>

In [1]:
import os
import subprocess
import json
import sys

# --- Choisir le profil de connexion ---
# 'local-native' : PostgreSQL installé localement
# 'local'        : PostgreSQL via Docker
CONNECTION_PROFILE = "local-native"

# --- Définir le mot de passe selon le profil ---
if CONNECTION_PROFILE == "local-native":
    os.environ["PYDBADMIN_NATIVE_PASSWORD"] = "postgres"
else:
    os.environ["PYDBADMIN_LOCAL_PASSWORD"] = "postgres"

# --- Chemin du fichier de config ---
CONFIG_PATH = "../config.toml"

print(f"Profil sélectionné : {CONNECTION_PROFILE}")
print(f"Config            : {os.path.abspath(CONFIG_PATH)}")
print(f"Python            : {sys.version}")

Profil sélectionné : local-native
Config            : c:\Users\awounfouet\Projects\packages\pydbadminkit\config.toml
Python            : 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]


In [2]:
def run(args: list[str], capture_json: bool = False):
    """Exécute une commande pydbadminkit et affiche le résultat."""
    base = [sys.executable, "-m", "pydbadminkit",
            "--connection", CONNECTION_PROFILE,
            "--config", CONFIG_PATH]
    cmd = base + args
    print("▶ " + " ".join(cmd[2:]))  # Affiche sans python -m
    print("-" * 60)
    result = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace")
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)
    if capture_json and result.stdout:
        try:
            return json.loads(result.stdout)
        except json.JSONDecodeError:
            return result.stdout
    return result.returncode

def run_json(args: list[str]):
    """Exécute et retourne le résultat parsé en JSON."""
    return run(["--output", "json"] + args, capture_json=True)

print("Fonctions utilitaires chargées.")

Fonctions utilitaires chargées.


---
## 2. Foundation — Connexion et capabilities <a id='2'></a>

In [3]:
# Version installée
result = subprocess.run(
    [sys.executable, "-m", "pydbadminkit", "--version"],
    capture_output=True, text=True
)
print(result.stdout.strip())

In [4]:
# Test de connexion
run(["connection", "test"])

▶ pydbadminkit --connection local-native --config ../config.toml connection test
------------------------------------------------------------
STDERR: c:\ProgramData\anaconda3\python.exe: No module named pydbadminkit



1

In [5]:
# Lister les connexions configurées dans config.toml
result = subprocess.run(
    [sys.executable, "-m", "pydbadminkit", "--config", CONFIG_PATH, "connection", "list"],
    capture_output=True, text=True, encoding="utf-8", errors="replace"
)
print(result.stdout)

In [6]:
# Capabilities de l'adaptateur
run(["capability"])

▶ pydbadminkit --connection local-native --config ../config.toml capability
------------------------------------------------------------
STDERR: c:\ProgramData\anaconda3\python.exe: No module named pydbadminkit



1

---
## 3. Object Explorer — Server & Databases <a id='3'></a>

In [ ]:
# Informations sur le serveur PostgreSQL
run(["server", "info"])

In [7]:
# Lister toutes les bases de données
run(["database", "list"])

▶ pydbadminkit --connection local-native --config ../config.toml database list
------------------------------------------------------------
STDERR: c:\ProgramData\anaconda3\python.exe: No module named pydbadminkit



1

In [8]:
# Décrire la base de développement
run(["database", "describe", "pydbadmin_dev"])

▶ pydbadminkit --connection local-native --config ../config.toml database describe pydbadmin_dev
------------------------------------------------------------
STDERR: c:\ProgramData\anaconda3\python.exe: No module named pydbadminkit



1

In [9]:
# Sortie JSON — pour exploitation programmatique
databases = run_json(["database", "list"])
print("Type retourné :", type(databases))
if isinstance(databases, list):
    print(f"{len(databases)} base(s) trouvée(s)")
    for db in databases:
        print(" -", db)

▶ pydbadminkit --connection local-native --config ../config.toml --output json database list
------------------------------------------------------------
STDERR: c:\ProgramData\anaconda3\python.exe: No module named pydbadminkit

Type retourné : <class 'int'>


---
## 4. Object Explorer — Schemas, Tables, Vues, Index <a id='4'></a>

In [ ]:
# Lister les schémas
run(["schema", "list"])

In [ ]:
# Schémas incluant les schémas système
run(["schema", "list", "--include-system"])

In [ ]:
# Créer les objets de démonstration dans la base
# (nécessite psycopg pour une connexion directe)
import psycopg

password = os.environ.get("PYDBADMIN_NATIVE_PASSWORD") or os.environ.get("PYDBADMIN_LOCAL_PASSWORD", "postgres")

sql_demo = """
CREATE TABLE IF NOT EXISTS public.customers (
    id BIGSERIAL PRIMARY KEY,
    email TEXT NOT NULL UNIQUE,
    name TEXT NOT NULL,
    created_at TIMESTAMPTZ NOT NULL DEFAULT now()
);

CREATE INDEX IF NOT EXISTS customers_name_idx ON public.customers(name);

CREATE OR REPLACE VIEW public.active_customers AS
SELECT id, email, name FROM public.customers;

INSERT INTO public.customers (email, name) VALUES
    ('alice@example.com', 'Alice Martin'),
    ('bob@example.com',   'Bob Dupont'),
    ('carol@example.com', 'Carol Lemaire')
ON CONFLICT (email) DO NOTHING;
"""

conn = psycopg.connect(
    host="localhost", port=5432,
    dbname="pydbadmin_dev", user="postgres",
    password=password
)
with conn:
    with conn.cursor() as cur:
        cur.execute(sql_demo)
conn.close()

print("Objets de démonstration créés : customers, customers_name_idx, active_customers")

In [ ]:
# Lister les tables du schema public
run(["table", "list", "--schema", "public"])

In [ ]:
# Décrire la table customers (colonnes, contraintes, PK, UNIQUE...)
run(["table", "describe", "public.customers"])

In [ ]:
# Lister les vues
run(["view", "list", "--schema", "public"])

In [ ]:
# Décrire la vue active_customers
run(["view", "describe", "public.active_customers"])

In [ ]:
# Lister les index
run(["index", "list", "--schema", "public", "--table", "customers"])

In [ ]:
# Décrire un index
run(["index", "describe", "public.customers_name_idx"])

---
## 5. Security — Inspection des rôles <a id='5'></a>

In [ ]:
# Lister tous les rôles
run(["role", "list"])

In [ ]:
# Seulement les rôles pouvant se connecter
run(["role", "list", "--login-only"])

In [ ]:
# Rôles incluant les rôles système PostgreSQL
run(["role", "list", "--include-system"])

In [ ]:
# Décrire le superuser postgres
run(["role", "describe", "postgres"])

---
## 6. Security — Accès directs et effectifs <a id='6'></a>

In [ ]:
# Accès directs du rôle postgres
run(["access", "list", "--role", "postgres"])

In [ ]:
# Accès effectifs du rôle postgres (inclut héritage, PUBLIC, superuser...)
run(["effective-access", "list", "--role", "postgres"])

In [ ]:
# Filtrer sur un objet spécifique
run(["access", "list", "--role", "postgres", "--schema", "public", "--object", "customers"])

---
## 7. Security — Ownership <a id='7'></a>

In [ ]:
# Objets possédés par postgres
run(["ownership", "list", "--owner", "postgres"])

In [ ]:
# Uniquement les tables du schema public
run(["ownership", "list", "--owner", "postgres", "--type", "table", "--schema", "public"])

---
## 8. Mutations — Dry-run & création de rôles <a id='8'></a>

> **Toujours commencer par `--dry-run`** pour inspecter le plan avant d'exécuter.

In [ ]:
# Dry-run : planifier la création d'un rôle (aucune mutation réelle)
run(["--dry-run", "role", "create", "demo_user", "--login"])

In [ ]:
# Dry-run en JSON — pour exploitation programmatique
plan = run_json(["--dry-run", "role", "create", "demo_user", "--login"])
print("Plan (JSON) :")
print(json.dumps(plan, indent=2, ensure_ascii=False) if isinstance(plan, (dict, list)) else plan)

In [ ]:
# Créer le rôle demo_user (mutation réelle avec --yes)
run(["--yes", "role", "create", "demo_user", "--login"])

In [ ]:
# Vérifier la création
run(["role", "describe", "demo_user"])

In [ ]:
# Créer un rôle de groupe (sans LOGIN)
run(["--yes", "role", "create", "demo_reader"])

In [ ]:
# Modifier demo_user : activer CREATEDB
run(["--yes", "role", "alter", "demo_user", "--createdb", "enable"])

In [ ]:
# Vérifier la modification
run(["role", "describe", "demo_user"])

In [ ]:
# Tester les guardrails — opération CRITIQUE doit être refusée avec --yes seul
print("=== Test guardrail : création superuser sans --confirm-target ===")
run(["--yes", "--non-interactive", "role", "create", "bad_super", "--superuser"])

In [ ]:
# Opération critique correcte : avec --confirm-target
print("=== Opération critique avec confirmation explicite ===")
run(["--non-interactive", "role", "create", "demo_super", "--superuser", "--confirm-target", "demo_super"])

---
## 9. Mutations — Memberships & GRANT/REVOKE <a id='9'></a>

In [ ]:
# Ajouter demo_user comme membre de demo_reader
run(["--yes", "role", "membership-add", "demo_reader", "demo_user"])

In [ ]:
# Vérifier le membership
run(["role", "describe", "demo_user"])

In [ ]:
# GRANT SELECT sur customers à demo_user
run(["--yes", "access", "grant",
     "--role", "demo_user",
     "--object", "public.customers",
     "--access", "SELECT"])

In [ ]:
# Vérifier l'ACL directe
run(["access", "list", "--role", "demo_user"])

In [ ]:
# Vérifier l'accès effectif (inclut héritage, ownership...)
run(["effective-access", "list", "--role", "demo_user"])

In [ ]:
# REVOKE SELECT
run(["--yes", "access", "revoke",
     "--role", "demo_user",
     "--object", "public.customers",
     "--access", "SELECT"])

In [ ]:
# Retirer le membership
run(["--yes", "role", "membership-remove", "demo_reader", "demo_user"])

---
## 10. Sortie machine — JSON & YAML <a id='10'></a>

In [ ]:
import yaml  # pip install pyyaml (inclus dans les dépendances du projet)

# Server info en JSON → exploitation Python
base = [sys.executable, "-m", "pydbadminkit",
        "--connection", CONNECTION_PROFILE,
        "--config", CONFIG_PATH,
        "--output", "json"]

result = subprocess.run(base + ["server", "info"],
                        capture_output=True, text=True, encoding="utf-8", errors="replace")
try:
    server_info = json.loads(result.stdout)
    print("Server info (JSON parsé) :")
    print(json.dumps(server_info, indent=2, ensure_ascii=False))
except Exception:
    print(result.stdout)

In [ ]:
# Database list en YAML
result = subprocess.run(
    [sys.executable, "-m", "pydbadminkit",
     "--connection", CONNECTION_PROFILE,
     "--config", CONFIG_PATH,
     "--output", "yaml",
     "database", "list"],
    capture_output=True, text=True, encoding="utf-8", errors="replace"
)
print("Database list (YAML) :")
print(result.stdout)

In [ ]:
# Rôles en JSON → filtrage Python
result = subprocess.run(
    [sys.executable, "-m", "pydbadminkit",
     "--connection", CONNECTION_PROFILE,
     "--config", CONFIG_PATH,
     "--output", "json",
     "role", "list", "--login-only"],
    capture_output=True, text=True, encoding="utf-8", errors="replace"
)
try:
    roles = json.loads(result.stdout)
    print(f"{len(roles)} rôle(s) avec LOGIN :")
    for r in roles:
        print(" -", r)
except Exception:
    print(result.stdout)

---
## 11. Nettoyage <a id='11'></a>

> Suppression des rôles et objets créés dans ce notebook.

In [ ]:
# Supprimer les rôles de démonstration
for role in ["demo_user", "demo_reader"]:
    print(f"Suppression du rôle : {role}")
    run(["--yes", "role", "drop", role])

# Supprimer demo_super si créé
run(["--non-interactive", "role", "drop", "demo_super", "--confirm-target", "demo_super"])

In [ ]:
# Supprimer les objets SQL de démonstration
password = os.environ.get("PYDBADMIN_NATIVE_PASSWORD") or os.environ.get("PYDBADMIN_LOCAL_PASSWORD", "postgres")

sql_cleanup = """
DROP VIEW  IF EXISTS public.active_customers;
DROP TABLE IF EXISTS public.customers;
"""

conn = psycopg.connect(
    host="localhost", port=5432,
    dbname="pydbadmin_dev", user="postgres",
    password=password
)
with conn:
    with conn.cursor() as cur:
        cur.execute(sql_cleanup)
conn.close()
print("Objets SQL supprimés : active_customers (view), customers (table)")

In [ ]:
# Vérification finale — état propre
print("=== État final des rôles ===")
run(["role", "list"])
print("\n=== Tables restantes ===")
run(["table", "list", "--schema", "public"])

---

## Récapitulatif des commandes

| Commande | Description |
|----------|-------------|
| `connection test` | Tester la connexion |
| `connection list` | Lister les profils configurés |
| `capability` | Afficher les capacités de l'adaptateur |
| `server info` | Informations sur le serveur |
| `database list / describe` | Explorer les bases de données |
| `schema list` | Explorer les schémas |
| `table list / describe` | Explorer les tables |
| `view list / describe` | Explorer les vues |
| `index list / describe` | Explorer les index |
| `role list / describe` | Inspecter les rôles |
| `access list` | ACL directes d'un rôle |
| `effective-access list` | Accès effectifs d'un rôle |
| `ownership list` | Objets possédés par un rôle |
| `role create / alter / drop` | Mutations de rôles |
| `role membership-add / remove` | Gestion des memberships |
| `access grant / revoke` | Gestion des ACL |

**Options globales utiles :**
- `--dry-run` : planifier sans exécuter
- `--yes` : approuver les confirmations simples
- `--non-interactive` : mode automatisé
- `--confirm-target <nom>` : confirmation typée pour opérations critiques
- `--output [table|json|yaml]` : format de sortie